## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-15-18-50-24 +0000,nyt,Chris Van Hollen Says His Majority Leader Chal...,https://www.nytimes.com/2026/11/15/us/politics...
1,2026-11-15-18-47-49 +0000,nyt,Israel Kidnapped Iranian Generals in Hunt for ...,https://www.nytimes.com/2026/11/15/world/europ...
2,2026-11-15-18-46-52 +0000,nyt,8 Takeaways From a Big Democratic Gathering in...,https://www.nytimes.com/2026/11/15/us/politics...
3,2026-11-15-18-44-16 +0000,nypost,"Iran still has stockpile of 2,500 missiles — a...",https://nypost.com/2026/11/15/world-news/iran-...
4,2026-11-15-18-44-00 +0000,wsj,Bessent Urges Nvidia to Pick Outsider as Next CEO,https://www.wsj.com/tech/ai/bessent-urges-nvid...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2457/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
72,ground,19
211,trump,18
101,new,16
23,iran,11
71,faa,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
29,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on nationwide ground stop...,https://www.cbc.ca/news/world/trump-faa-ground...,71
51,2026-11-15-15-37-44 +0000,wapo,Trump blames Biden for FAA problems in first c...,https://www.washingtonpost.com/politics/2026/1...,71
47,2026-11-15-15-52-33 +0000,nypost,Sean Duffy reveals when airport travel delays ...,https://nypost.com/2026/11/15/us-news/sean-duf...,69
12,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...,62
17,2026-11-15-18-06-20 +0000,nyt,Sean Duffy Offers New Details on How One Fix S...,https://www.nytimes.com/2026/11/15/us/sean-duf...,58


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
29,71,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on nationwide ground stop...,https://www.cbc.ca/news/world/trump-faa-ground...
12,62,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...
221,39,2026-11-14-20-00-00 +0000,wapo,Chris Van Hollen revs up his public campaign t...,https://www.washingtonpost.com/politics/2026/1...
60,37,2026-11-15-15-00-00 +0000,wapo,"As Duffy seeks air traffic control upgrades, D...",https://www.washingtonpost.com/politics/2026/1...
136,36,2026-11-15-05-51-30 +0000,nypost,Bahrain issues first threat alerts since Septe...,https://nypost.com/2026/11/15/world-news/iran-...
156,32,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...
218,32,2026-11-14-20-08-07 +0000,cbc,New Orleans port sues for control of China-lin...,https://www.cbc.ca/news/world/new-orleans-tank...
147,31,2026-11-15-03-27-33 +0000,nypost,Hundreds gather to remember ‘loving’ young con...,https://nypost.com/2026/11/14/us-news/hundreds...
107,29,2026-11-15-10-30-00 +0000,wsj,High mortgage rates have crushed homeowners’ b...,https://www.wsj.com/economy/housing/theyre-luc...
103,29,2026-11-15-10-34-22 +0000,nypost,Vagrant sets up City Hall encampment along Mam...,https://nypost.com/2026/11/15/us-news/mamdanis...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
